# 機能テスト 2: Topic Graph(`topicgraph` ネイティブ実行ファイル)

## サマリー

- Topic Graph を**作る主体は、`Azure/foundry-iq-tool-releases` で配布されるネイティブ実行ファイル `topicgraph`**(.NET)です。foundry-iq-skills は、リリース v1.0.10 を SHA-256 で固定して使います。実行ファイルは、**利用者のマシンで**、Foundry IQ のインデックスに保存されたチャンク ベクトルを読み、トピックの階層を作ります。プラグインの Python は、設定 JSON を書いて起動し、出力を読むだけです。このノートブックは、その実行ファイルを**直接**起動し、Foundry IQ やモデルとの連携を含めて機能をテストします。
- 入力は、同じフォルダーにある 2 つのテキスト(`近藤寺 - Hanapedia.txt`、`花ケ咲神社 - Hanapedia.txt`)です。ノートブックの中で、チャンクに分け、埋め込みを作り、**テスト専用の小さなソース インデックス**(`ft-topic-src`)に入れます。他の機能テスト(Entity Graph)には依存しません。
- 実行ファイルは、ビルドの間だけ、**ソース インデックスに一時的なスコアリング プロファイルを追加し、終了時に削除します**(文書とフィールドは変えません)。この「元に戻ること」も、テストします。
- 出力は `.functional-test/topicgraph-out` に残ります。**機能テスト 3(オントロジー)の入力**として使えます。
- **9 章では Foundry IQ への登録・読み戻し・実データの可視化・GQL 探索まで実行できます。** `RUN_FOUNDRY_IQ = True` で有効にし、初回書き込みは `CONFIRM_TOPIC_WRITES` で別途承認します。他のノートブックには依存しません。
- セクション 6.5 で**全トピックの親子関係をグラフ表示**し、6.6 で**階層と所属チャンク数の割合をトピックホイールで表示**します。どちらも生成済みのデータを使い、追加のモデル呼び出しや Foundry IQ への書き込みは行いません。
- **既存出力があれば、既定で読み取り専用の再開モードになります。** `Run All` でもインデックスの再作成、埋め込み生成、ビルド、後片付けは実行しません。新規作成する場合だけ、1.2 の `USE_EXISTING_OUTPUT` を `False` にします。

### 既存出力からのクイックスタート

1. カーネルを再起動した場合は、**1.2 の設定セル → 5.0 の出力読み込みセル**を実行します。初回だけセクション 1 のインストールセルも実行してください。
2. **5.1 と 6.0～6.6** で、保存済みの実行記録・所属・トピックグラフ・ホイールを確認できます。グラフだけなら、5.0 の次に 6.5 → 6.6 を直接実行できます。
3. **10** で、今回実際に行った検証の結果を確認します。新規作成時だけのテストを、再実行したことにはしません。

必要な既存出力は `.functional-test/topicgraph-out/topicGraph` です。ローカル出力の読み込みには `.env`、Azure への認証、元のテキスト、実行ファイルは不要です。古い出力に元チャンクの本文・埋め込みがない場合は、6.2 ではチャンク ID を表示し、6.3 のチャンク PCA は実行しません。新しく作成するビルドでは、再開用にそれらも保存します。

> 出典: [Azure/foundry-iq-tool-releases(TopicGraph のリリース)](https://github.com/Azure/foundry-iq-tool-releases) / [Topic Graph build: Plan and apply(foundry-iq-skills)](https://github.com/microsoft/azure-skills/blob/main/.github/plugins/foundry-iq-skills/skills/foundry-iq/workflows/graph/topic-graph-build/references/plan-and-apply.md) / [Topic Graph build(foundry-iq-skills)](https://github.com/microsoft/azure-skills/blob/main/.github/plugins/foundry-iq-skills/skills/foundry-iq/workflows/graph/topic-graph-build/topic-graph-build.md)

## 作成されるリソースと関連コンポーネント

**ノートブックがソース インデックスを作成し、利用者のマシンで動く `topicgraph` がトピックの構造と出力ファイルを生成します。** Entity Graph のように、Foundry IQ がナレッジ ソースから取り込み用コンポーネントを自動生成する構成ではありません。青はノートブックが作成・実行するもの、緑は `topicgraph` が生成するもの、灰色は既存の入力・サービスと配布済みの実行ファイルです。実線は主な処理・データの流れ、破線は設定・モデル呼び出し・一時的な変更を表します。

```mermaid
flowchart TB
    subgraph LOCAL["利用者のマシン"]
        TXT["既存の 2 記事テキスト"]
        NB["ノートブック<br/>T1: 整形・チャンク化・埋め込み取得<br/>T2: インデックスを作成"]
        CFG["設定ファイル<br/>topicgraph-config.json<br/>T3: ノートブックが保存"]
        CLI["topicgraph (.NET 固定版)<br/>T3: preflight / T4: build<br/>T7: build --resume"]
        subgraph OUT["ローカル出力: OUT"]
            STRUCT["トピック階層・所属・ラベル<br/>topicGraph の構造・ラベル<br/>ontology へ構造・ラベルを複製"]
            EXPORT["エクスポート<br/>releases<br/>スキーマ・グラフ文書"]
            STATE["実行記録・チェックポイント<br/>build-summary.json<br/>run-manifest.json"]
        end
        TXT --> NB
        CFG -. 実行設定を読み込み .-> CLI
        CLI --> STRUCT
        CLI --> EXPORT
        CLI <-->|進捗の保存・再開| STATE
    end

    subgraph SEARCH["Foundry IQ (既存のサービス)"]
        IDX[("ソース インデックス<br/>ft-topic-src<br/>本文 + snippet_vector")]
        TI[("公開用トピック インデックス<br/>9 章: 登録・読み戻し・GQL")]
        PROFILE["一時スコアリング プロファイル<br/>ソース インデックスの定義内<br/>T6: 削除・復元を検証"]
    end
    subgraph AOAI["Azure OpenAI (既存のモデル デプロイ)"]
        EMB["埋め込みモデル<br/>T1: チャンクのベクトル化"]
        CHAT["チャット モデル<br/>ステージ 7: トピックのラベル付け"]
    end

    EXPORT -->|9 章: 明示的な承認後に登録| TI
    TI --> READ["登録済みの階層を可視化・探索"]
    NB -->|T2: 本文・ベクトルをアップロード| IDX
    IDX -->|チャンク・ベクトル・近傍検索結果| CLI
    NB -. 埋め込みを依頼 .-> EMB
    CLI -. タイトル・説明の生成を依頼 .-> CHAT
    CLI -. ビルド中に追加・終了時に削除 .-> PROFILE

    classDef notebook fill:#dbeafe,stroke:#2563eb,color:#172554
    classDef generated fill:#dcfce7,stroke:#16a34a,color:#14532d
    classDef existing fill:#f1f5f9,stroke:#64748b,color:#0f172a
    class NB,CFG,IDX,TI,READ notebook
    class STRUCT,EXPORT,STATE,PROFILE generated
    class TXT,CLI,EMB,CHAT existing
    style PROFILE stroke-dasharray:5 5
```

- **実行場所と生成主体:** `topicgraph` は T0 で固定版を取得または再利用し、SHA-256 を検証した実行ファイルです。トピックの構築はローカルで行い、近傍の取得には Foundry IQ、ラベルの生成にはチャット モデルを呼び出します。一時的なスコアリング プロファイルは Foundry IQ の独立したリソースではなく、`ft-topic-src` の定義に追加されます。
- **このテストで作成しないもの:** ナレッジ ソース、ナレッジ ベース、データ ソース、インデクサー、スキルセットは作成しません。本文とベクトルは SDK で直接アップロードします。`blob_url` は文書の識別に使うフィールド名で、このテストは記事を Blob Storage にアップロードしません。
- **出力と後続処理:** `OUT` は `.functional-test\topicgraph-out` です。`topicGraph` 配下に構造・ラベル・エクスポート・チェックポイントを保存し、`ontology` 配下の構造とラベルを機能テスト 3 に渡せます。チェックポイントはローカルの再開用ファイルで、エンリッチメント キャッシュではありません。**ローカル確認だけならグラフ文書はアップロードしません。** 9 章で明示的に有効にした場合は、専用トピック インデックスにアップロードして読み戻します。検索プロファイルへの接続は別の手順です。

> 出典: [Topic Graph のネイティブ実行ファイルと検証](https://github.com/microsoft/azure-skills/blob/main/.github/plugins/foundry-iq-skills/skills/foundry-iq/workflows/graph/topic-graph-build/references/native-runtime.md) / [Topic Graph の構築とインデックスへのアップロードの区別](https://github.com/microsoft/azure-skills/blob/main/.github/plugins/foundry-iq-skills/skills/foundry-iq/workflows/graph/topic-graph-build/references/plan-and-apply.md) / [Topic Graph を検索に接続する前提と手順](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-topic-graph)

## テストの流れ

```mermaid
flowchart LR
    TXT["2 つのテキスト"] --> CH["チャンク分割"] --> EMB["埋め込み<br/>Azure OpenAI"] --> IDX[("ソース インデックス<br/>ft-topic-src")]
    IDX --> PF["preflight<br/>読み取り専用"] --> BD["build --fresh<br/>ステージ 0 から 9"] --> OUT[("出力フォルダー")]
    BD -. "一時的なスコアリング<br/>プロファイル" .-> IDX
    OUT --> RD["結果の読み取り<br/>トピック / 所属 / ラベル"]
    OUT --> RS["build --resume<br/>再利用の確認"]
    OUT --> PUB["承認後に Foundry IQ へ登録"] --> LIVE["読み戻し・可視化・GQL"]
```

## テスト一覧

| ID | 対象 | 合格条件 |
| --- | --- | --- |
| T0 | 実行ファイル | 見つかる(または固定版をダウンロードできる)。`--version` が固定版と一致する。SHA-256 がプラグインの記録と一致する |
| T1 | 入力の用意 | チャンクが 20 個以上できる。埋め込みの次元がそろう |
| T2 | ソース インデックス | 文書数が一致する。ベクトル フィールドが、保存され、取得可能で、HNSW のコサイン類似度である |
| T3 | `preflight` | 成功する。文書数と次元が一致する。一時的な変更が予告される |
| T4 | `build` | 初回は `--fresh`、中断したビルドは `--resume` で成功する。ステージ 0 から 9 がすべて完了または再利用される。kNN の累積呼び出し回数がチャンク数と一致する。モデルが呼ばれる。失敗した呼び出しがない |
| T5 | 出力の中身 | 最新のリリースが `exported`。すべてのチャンクがトピックに入る。末端トピックが 2 つ以上。所属の合計がチャンク数。すべてのトピックにタイトルが付く |
| T6 | 一時的な変更の復元 | スコアリング プロファイルが削除され、文書数とベクトル フィールドの設定が変わらない |
| T7 | `build --resume` | 完了した build を再利用し、モデルを再び呼ばず、リリースが変わらない |

## 前提

| 項目 | 内容 |
| --- | --- |
| 認証 | Azure CLI でサインイン済みであること(`az login`)。実行ファイルは `defaultAzureCredential`(対話ブラウザなし)で、Foundry IQ とモデルを呼びます |
| ロール(目安) | Foundry IQ: Search Service Contributor(インデックスの作成と、スコアリング プロファイルの一時追加)、Search Index Data Contributor(文書のアップロード)。Azure OpenAI: Cognitive Services OpenAI User |
| `.env` | `AZURE_SEARCH_ENDPOINT`、`AZURE_OPENAI_ENDPOINT`、`AZURE_OPENAI_DEPLOYMENT`、`AZURE_OPENAI_EMBEDDING_DEPLOYMENT`(任意で `AZURE_OPENAI_MODEL`) |
| OS | Windows x64 / arm64 で確認する想定です。Linux と macOS では、環境変数 `TOPICGRAPH_EXE` に実行ファイルのパスを指定します(プラグインが対応するのは、Windows と Linux です) |

## 1. セットアップ

### 1.1 パッケージのインストール

ナレッジ グラフのモデルと API を含む `azure-search-documents` の 12.2.0b1 を使います(インデックスの作成と文書のアップロードに使います)。

In [ ]:
%pip install -q azure-search-documents==12.2.0b1 azure-identity python-dotenv pandas matplotlib requests networkx plotly

### 1.2 設定

`check()` は、テストの結果を `results` に記録します。失敗しても後続のセルは続けて実行し、最後にまとめて表示します。

主なパラメーターは次のとおりです。

| 名前 | 意味 |
| --- | --- |
| `USE_EXISTING_OUTPUT` | 出力フォルダーがあれば自動で `True` にします。読み取り専用で再開します。明示的に `False` にすると、インデックスの削除・再作成を含む作成経路を実行します |
| `OUT` | 保存済み出力、または新規ビルドの出力先。既定はノートブック配下の `.functional-test/topicgraph-out` です |
| `CHUNK_CHARS` | チャンクの目安の文字数。小さいほどチャンクが増え、トピックの分割を確かめやすくなります |
| `MAX_LEAF_SIZE` | 末端トピックの大きさの目標(Leiden による再分割の目標、最小 2)。既定は 100 ですが、2 記事のチャンクは 50 個ほどなので、小さくして階層を作らせます |

In [ ]:
import base64
import hashlib
import json
import os
import platform
import shutil
import struct
import subprocess
import time
import zipfile
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv
from IPython.display import display

matplotlib.rcParams["font.sans-serif"] = ["Yu Gothic", "Meiryo", "MS Gothic", "Noto Sans CJK JP", "Noto Sans JP", "Hiragino Sans", "DejaVu Sans"]

FT = "ft-topic"
SRC_INDEX = f"{FT}-src"
NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "functional-test-2-topic-graph.ipynb").is_file() and (NOTEBOOK_DIR / "graph" / "functional-test-2-topic-graph.ipynb").is_file():
    NOTEBOOK_DIR = NOTEBOOK_DIR / "graph"
WORK = (NOTEBOOK_DIR / ".functional-test").resolve()
OUT = WORK / "topicgraph-out"      # 機能テスト 3 が、既定でここを探します
USE_EXISTING_OUTPUT = (OUT / "topicGraph").is_dir()
CHUNK_CHARS = 250
MAX_LEAF_SIZE = 8
RUN_CLEANUP = False                # True にすると、最後にインデックスと作業フォルダーを削除します

results = []
if not USE_EXISTING_OUTPUT:
    from azure.identity import DefaultAzureCredential

    load_dotenv(NOTEBOOK_DIR / ".env", override=True)
    credential = DefaultAzureCredential()
    SEARCH_ENDPOINT = os.environ["AZURE_SEARCH_ENDPOINT"].rstrip("/")
    AOAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/")
    CHAT_DEPLOYMENT = os.environ["AZURE_OPENAI_DEPLOYMENT"]
    CHAT_MODEL = os.environ.get("AZURE_OPENAI_MODEL", CHAT_DEPLOYMENT)
    EMBED_DEPLOYMENT = os.environ["AZURE_OPENAI_EMBEDDING_DEPLOYMENT"]
    WORK.mkdir(exist_ok=True)


def check(case, ok, detail=""):
    results.append({"テスト": case, "結果": "PASS" if ok else "FAIL", "詳細": str(detail)[:200]})
    print("PASS" if ok else "FAIL", "-", case, f"| {detail}" if detail else "")
    return ok


print("既存出力の読み込み" if USE_EXISTING_OUTPUT else "新規作成・未完了ビルドの再開", "|", OUT)
print("次は 5.0 の出力読み込みセルです。" if USE_EXISTING_OUTPUT else "セクション 2～5 で入力とビルドを作成します。")

## 2. T0: 実行ファイルの確認

`topicgraph` は、自己完結型の .NET の実行ファイルです。プラグインは、`native-release.json` に、リポジトリ(`Azure/foundry-iq-tool-releases`)、タグ(`v1.0.10`)、プラットフォームごとの資産の SHA-256 を固定しています。取得した配布物のダイジェストを照合してから展開し、展開した実行ファイルの SHA-256 を `receipt.json` に記録します。

ここでは、次の順で実行ファイルを探します。

1. 環境変数 `TOPICGRAPH_EXE`(任意のパス)
2. プラグインのキャッシュ(`%LOCALAPPDATA%\foundry-iq-skills\runtimes\topicgraph\native\win-x64\v*\extracted\topicgraph.exe`)
3. なければ、固定版の `win-x64` を GitHub Releases からダウンロードし、**固定されたダイジェストと照合**して展開します

`topicgraph --version` は、設定も認証もサービスへのアクセスも要りません。

> 出典: [Azure/foundry-iq-tool-releases](https://github.com/Azure/foundry-iq-tool-releases) / [native-release.json と native_download.py(topicgraph_skills ホイール)](https://github.com/microsoft/azure-skills/tree/main/.github/plugins/foundry-iq-skills/graph/runtime/wheels)

In [ ]:
if USE_EXISTING_OUTPUT:
    print("既存出力モード: 実行ファイルの取得・起動は行いません。")
else:
    PIN_VERSION = "1.0.10"
    PIN_URL = f"https://github.com/Azure/foundry-iq-tool-releases/releases/download/v{PIN_VERSION}/topicGraphBuildTool-{PIN_VERSION}-win-x64.zip"
    PIN_SHA256 = "b0873e104f79f021646ba310b769dcfd21435ad2f30759f7c2be76ead0661849"   # foundry-iq-skills の native-release.json(win-x64)

    cache_root = Path(os.environ.get("FOUNDRY_IQ_RUNTIME_CACHE") or (Path(os.environ.get("LOCALAPPDATA", Path.home() / ".cache")) / "foundry-iq-skills" / "runtimes"))
    EXE = Path(os.environ["TOPICGRAPH_EXE"]) if os.environ.get("TOPICGRAPH_EXE") else None
    if EXE is None:
        cached = sorted(cache_root.glob("topicgraph/native/win-x64/v*/extracted/topicgraph.exe"))
        EXE = cached[-1] if cached else None

    if EXE is None:
        assert platform.system() == "Windows", "TOPICGRAPH_EXE に、topicgraph 実行ファイルのパスを指定してください"
        archive = WORK / "native" / f"topicGraphBuildTool-{PIN_VERSION}-win-x64.zip"
        archive.parent.mkdir(parents=True, exist_ok=True)
        archive.write_bytes(requests.get(PIN_URL, timeout=300).content)
        assert hashlib.sha256(archive.read_bytes()).hexdigest() == PIN_SHA256, "ダウンロードした配布物のダイジェストが、固定された値と一致しません"
        zipfile.ZipFile(archive).extractall(archive.parent / "extracted")
        EXE = archive.parent / "extracted" / "topicgraph.exe"

    check("T0-1 実行ファイルがある", EXE.is_file(), EXE)
    version = subprocess.run([str(EXE), "--version"], capture_output=True, text=True, encoding="utf-8", errors="replace").stdout.strip()
    check("T0-2 --version が固定版と一致する", version == PIN_VERSION, version)

    receipt_path = EXE.parent.parent / "receipt.json"
    actual_sha = hashlib.sha256(EXE.read_bytes()).hexdigest().upper()
    if receipt_path.is_file():
        recorded = json.loads(receipt_path.read_text(encoding="utf-8"))["binary_sha256"].upper()
        check("T0-3 実行ファイルの SHA-256 がプラグインの記録と一致する", actual_sha == recorded, actual_sha[:16])
    else:
        print("プラグインの記録(receipt.json)がないため、SHA-256 の照合は省略しました。実行ファイルの SHA-256:", actual_sha)

## 3. 入力の用意

### 3.1 なぜ、自前のソース インデックスを作るのか

`topicgraph` が読むのは、**Foundry IQ のインデックスに保存されたチャンクのベクトル**です。要件は、次のとおりです(`topicgraph --help` と、プラグインの `native_source_policy`)。

- ベクトル フィールドが、**保存され(stored)、取得可能(retrievable)な float32** で、HNSW のコサイン類似度(または内積)であること。
- キー フィールド(`uid`)があること。本文(`snippet`)は、ラベル付けのために指定します。文書の出所(`blob_url`)は、任意の追跡用の項目です。

Entity Graph のナレッジ ソースが作るコンテンツ インデックスも、この条件を満たします。ただし、このテストは Entity Graph に依存しないよう、**同じ形のインデックスを、テキストから自作します**。HNSW は、近傍探索のためのグラフ型の索引です。

> 出典: [Efficient and robust approximate nearest neighbor search using HNSW graphs(Malkov and Yashunin)](https://arxiv.org/abs/1603.09320) / [Plan and apply(foundry-iq-skills)](https://github.com/microsoft/azure-skills/blob/main/.github/plugins/foundry-iq-skills/skills/foundry-iq/workflows/graph/topic-graph-build/references/plan-and-apply.md)

### 3.2 テキストの整形とチャンク分割

2 つのファイルの拡張子は `.txt` ですが、内容は見出し・段落・表を使った Markdown です。改行は実際の改行として保存されており、ページ情報の HTML コメントやエスケープされた改行はありません。既存の整形処理を通し、段落を `CHUNK_CHARS` を目安にまとめてチャンクを作ります。

In [ ]:
import re

DATA_DIR = NOTEBOOK_DIR
SAMPLES = {"近藤寺 - Hanapedia.txt": "kondoji", "花ケ咲神社 - Hanapedia.txt": "hanagasaki-jinja"}


def clean_text(raw):
    text = raw.replace("\\n", "\n").replace('\\"', '"')
    text = re.sub(r"<!--.*?-->", "", text, flags=re.S)
    text = re.sub(r"<tr>(.*?)</tr>", lambda m: " / ".join(re.findall(r"<td>(.*?)</td>", m.group(1), flags=re.S)) + "\n", text, flags=re.S)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"(?<=[\u3000-\u9fff、。」』)])\n(?=[\u3000-\u9fffA-Za-z「『])", "", text)
    text = re.sub(r"[ \t]{2,}", " ", text)
    return re.sub(r"\n{3,}", "\n\n", text).strip()


def chunk_text(text, target):
    chunks, buf = [], ""
    for para in re.split(r"\n\s*\n", text):
        para = para.strip()
        if para and buf and len(buf) + len(para) > target:
            chunks.append(buf.strip())
            buf = ""
        buf += para + "\n\n" if para else ""
    return chunks + ([buf.strip()] if buf.strip() else [])


if USE_EXISTING_OUTPUT:
    print("既存出力モード: 元テキストの読み込み・再チャンク化は行いません。")
else:
    chunks = []
    for src, slug in SAMPLES.items():
        for n, body in enumerate(chunk_text(clean_text((DATA_DIR / src).read_text(encoding="utf-8")), CHUNK_CHARS)):
            chunks.append({"uid": f"{FT}-{slug}-{n:03d}", "blob_url": f"{slug}.txt", "snippet": body})

    chunk_df = pd.DataFrame(chunks)
    check("T1-1 チャンクが 20 個以上できる", len(chunk_df) >= 20, f"{len(chunk_df)} 個")
    check("T1-2 空のチャンクがない", bool((chunk_df["snippet"].str.len() > 0).all()), f"最短 {chunk_df['snippet'].str.len().min()} 文字")
    display(chunk_df.groupby("blob_url")["snippet"].agg(チャンク数="count", 平均文字数=lambda s: round(s.str.len().mean())))
    display(chunk_df.head(3))

### 3.3 埋め込みの作成

Azure OpenAI の埋め込みデプロイメントで、チャンクをベクトルにします(16 個ずつ)。ベクトルの次元は、モデルが決めるので、応答から読み取ります。この後の可視化(主成分分析)にも、このベクトルを使います。

In [ ]:
def embed(texts):
    token = credential.get_token("https://cognitiveservices.azure.com/.default").token
    response = requests.post(f"{AOAI_ENDPOINT}/openai/deployments/{EMBED_DEPLOYMENT}/embeddings?api-version=2024-10-21",
                             headers={"Authorization": f"Bearer {token}"}, json={"input": texts}, timeout=120)
    response.raise_for_status()
    return [d["embedding"] for d in sorted(response.json()["data"], key=lambda d: d["index"])]


if USE_EXISTING_OUTPUT:
    print("既存出力モード: 埋め込みは生成しません。")
else:
    vectors = [v for i in range(0, len(chunks), 16) for v in embed([c["snippet"] for c in chunks[i:i + 16]])]
    DIMS = len(vectors[0])
    check("T1-3 すべての埋め込みの次元がそろう", all(len(v) == DIMS for v in vectors) and len(vectors) == len(chunks), f"{len(vectors)} 本 x {DIMS} 次元")

### 3.4 T2: ソース インデックスの作成

`uid` をキー(フィルター可能)、`snippet` を全文検索可能な本文、`snippet_vector` を、**保存され、取得可能**なベクトル フィールド(HNSW、コサイン)にします。同じ名前のインデックスが前回の実行で残っていれば、削除して作り直します(この名前は、このテスト専用です)。作成したら、設定が意図どおりに保存されたかを、読み戻して確かめます。

In [ ]:
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    HnswAlgorithmConfiguration,
    HnswParameters,
    SearchField,
    SearchFieldDataType,
    SearchIndex,
    VectorSearch,
    VectorSearchAlgorithmMetric,
    VectorSearchProfile,
)

if USE_EXISTING_OUTPUT:
    print("既存出力モード: インデックスの削除・作成・アップロードは行いません。")
else:
    index_client = SearchIndexClient(SEARCH_ENDPOINT, credential)
    if SRC_INDEX in list(index_client.list_index_names()):
        index_client.delete_index(SRC_INDEX)

    index_client.create_index(SearchIndex(
        name=SRC_INDEX,
        fields=[
            SearchField(name="uid", type=SearchFieldDataType.String, key=True, filterable=True),
            SearchField(name="blob_url", type=SearchFieldDataType.String, filterable=True),
            SearchField(name="snippet", type=SearchFieldDataType.String, searchable=True),
            SearchField(name="snippet_vector", type=SearchFieldDataType.Collection(SearchFieldDataType.Single), searchable=True,
                        vector_search_dimensions=DIMS, vector_search_profile_name="vp", hidden=False, stored=True),
        ],
        vector_search=VectorSearch(
            algorithms=[HnswAlgorithmConfiguration(name="hnsw", parameters=HnswParameters(metric=VectorSearchAlgorithmMetric.COSINE))],
            profiles=[VectorSearchProfile(name="vp", algorithm_configuration_name="hnsw")]),
    ))

    search_client = SearchClient(SEARCH_ENDPOINT, SRC_INDEX, credential)
    search_client.upload_documents([{**c, "snippet_vector": v} for c, v in zip(chunks, vectors)])
    for _ in range(30):  # 文書数の反映は、少し遅れることがあります
        if search_client.get_document_count() == len(chunks):
            break
        time.sleep(2)

    index = index_client.get_index(SRC_INDEX)
    vector_field = next(f for f in index.fields if f.name == "snippet_vector")
    metric = index.vector_search.algorithms[0].parameters.metric
    before = {"docs": search_client.get_document_count(), "scoring_profiles": [p.name for p in (index.scoring_profiles or [])],
              "vector_flags": (vector_field.stored, vector_field.hidden)}

    check("T2-1 文書数がチャンク数と一致する", before["docs"] == len(chunks), f"{before['docs']} / {len(chunks)}")
    check("T2-2 ベクトル フィールドが、保存され、取得可能で、HNSW のコサインである",
          vector_field.stored is not False and not vector_field.hidden and vector_field.vector_search_dimensions == DIMS and getattr(metric, "value", metric) == "cosine",
          f"stored={vector_field.stored} hidden={vector_field.hidden} metric={getattr(metric, 'value', metric)}")

## 4. 設定ファイルと T3: `preflight`

実行ファイルは、**入れ子の設定 JSON**(`--config`)で動かします。プラグインが書く設定と同じ形です。

| セクション | 内容 |
| --- | --- |
| `search` | エンドポイント、API バージョン(`2026-11-01-preview`)、ソース インデックス、ベクトル、本文、文書の出所のフィールド |
| `authentication` | `defaultAzureCredential`、対話ブラウザなし、テナント ID |
| `models` | ラベル付けに使う、Azure OpenAI のチャット デプロイメント(**最大 1 つ**) |
| `execution.topicGraph` | サンプリング、`maxLeafSize`、並列度、秒間リクエスト数。`readOnly: false` は、一時的なスコアリング プロファイルを許すためです |
| `execution.sitemap` | `enabled: true` で、ステージ 7(ラベル付け)を有効にします |
| `outputDirectory` | 出力の場所。実行ファイルは、この下の `topicGraph` を所有します |

`preflight` は、**読み取り専用**の確認で、`build --dry-run` と同じです。ソースのスキーマ、文書数、次元を調べ、呼び出し回数を見積もり、**これから行う一時的な変更**を予告します。

出力フォルダーは削除せず、中断時のチェックポイントを残します。未完了のビルドがあれば、T4 で同じ設定と出力先を使って再開します。

> 出典: [RUN-TOPICGRAPH.md(配布物に同梱)](https://github.com/Azure/foundry-iq-tool-releases) / [native_adapter.py(topicgraph_skills ホイール)](https://github.com/microsoft/azure-skills/tree/main/.github/plugins/foundry-iq-skills/graph/runtime/wheels)

In [ ]:
def run_exe(*args, timeout=None):
    return subprocess.run([str(EXE), *map(str, args)], capture_output=True, text=True, encoding="utf-8", errors="replace", timeout=timeout)


if USE_EXISTING_OUTPUT:
    print("既存出力モード: 設定の上書き・認証・preflight は行いません。")
else:
    payload = credential.get_token("https://management.azure.com/.default").token.split(".")[1]
    TENANT_ID = json.loads(base64.urlsafe_b64decode(payload + "=" * (-len(payload) % 4)))["tid"]
    OUT.mkdir(parents=True, exist_ok=True)
    CONFIG_PATH = OUT / "topicgraph-config.json"
    config = {
        "search": {"endpoint": SEARCH_ENDPOINT, "apiVersion": "2026-11-01-preview", "sourceIndex": SRC_INDEX,
                   "vectorField": "snippet_vector", "textField": "snippet", "docIdField": "blob_url"},
        "authentication": {"mode": "defaultAzureCredential", "allowInteractiveBrowser": False, "tenantId": TENANT_ID},
        "models": [{"kind": "azureOpenAI", "azureOpenAIParameters": {
            "resourceUri": AOAI_ENDPOINT, "deploymentId": CHAT_DEPLOYMENT, "modelName": CHAT_MODEL, "modelApiVersion": "2024-12-01-preview"}}],
        "execution": {
            "topicGraph": {"samplingPercentage": 4, "minimumSampleCount": 30000, "probeSeed": 1234, "maxLeafSize": MAX_LEAF_SIZE,
                           "concurrency": 8, "requestsPerSecond": 25, "readOnly": False},
            "sitemap": {"enabled": True, "label_title_words": 8, "label_description_words": 60, "label_representatives": 4, "label_max_completion_tokens": 4096},
        },
        "outputDirectory": str(OUT),
    }
    CONFIG_PATH.write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8")
    pre = run_exe("preflight", "--config", CONFIG_PATH, timeout=600)
    print(pre.stdout[-1500:], pre.stderr)
    pre.check_returncode()
    report = json.loads((OUT / "topicGraph" / "step-00-preflight" / "report.json").read_text(encoding="utf-8"))

    check("T3-1 preflight が成功する", pre.returncode == 0, f"終了コード {pre.returncode}")
    check("T3-2 文書数と次元がソースと一致する", report["documentCount"] == len(chunks) and report["dimensions"] == DIMS, f"{report['documentCount']} 件 / {report['dimensions']} 次元")
    check("T3-3 ベクトル フィールドが取得可能と判定される", report["vectorFieldRetrievable"] is True)
    check("T3-4 一時的なスコアリング プロファイルの追加が予告される", any("scoring profile" in m for m in report["plannedMutations"]), report["plannedMutations"][0][:80])
    display(pd.DataFrame([report["callEstimate"]]).T.rename(columns={0: "見積もり"}))

## 5. T4: ビルドと中断時の再開

`build` は、**ステージ 0 から 9** を順に実行します。`--fresh` は、新しい世代を始めます(成功した前の世代だけを残します)。途中で止まっても、確定済みの成果(チェックポイント)から `--resume` で再開できます。

このセルは、未完了の `build-summary.json` があれば `--resume`、なければ `--fresh` を選びます。Windows でステージ 6 の構造保存が `System.IO.IOException` のファイル共有違反で止まった場合だけ、2 秒、4 秒待って、最大 2 回 `--resume` で再試行します。その他のエラーや再試行上限では、標準エラー全文を表示して例外を送出し、後続の出力読み取りには進みません。ロックが続く場合は、そのファイルを使用しているアプリを確認してから、T4 を再実行してください。ロック元のプロセスを、このノートブックが終了させることはありません。

再開時のステージ状態は `completed` または `reused` です。`calls` は、チェックポイントに保存された累積回数で検証します。復旧時は T3 で出力を消したり、T2 でソース インデックスを作り直したりせず、T4 から再実行します。

| ステージ | 内容 |
| --- | --- |
| 0 | Preflight: スキーマ、文書数、次元の確認 |
| 1 | Probes: 各チャンクを起点(アンカー)に、ベクトル検索で近傍(kNN)を取る。アンカーの数は `min(N, max(最小サンプル数, ceil(N x サンプリング率 / 100)))` |
| 2 | Views: kNN グラフの候補(`k` と類似度の閾値)を作り、診断の基準(最大連結成分の割合と平均次数)で選ぶ |
| 3 | Clustering: **Leiden 法**(モジュラリティ)で分割する。`maxLeafSize` が、末端の大きさの目標 |
| 4 | Hierarchy: 分割を、トピックの親子の階層に整理する |
| 5 | Leaves: 末端トピックごとに、ベクトルの統計(重心)と代表チャンクを取得する |
| 6 | Structure: 構造を封印(sealed)して保存する |
| 7 | Labeling: チャット モデルで、トピックのタイトルと説明を作る |
| 8 | Derive: 割合などのグラフの属性を導出する |
| 9 | Export: グラフのスキーマと文書(トリプル)を書き出す |

この表は、配布物のヘルプと、過去の実行のログに基づきます。クラスタリング自体の実装は実行ファイルの内部で、ソースが公開されていないため、アルゴリズムの細部は確認できません。Leiden 法は、Louvain 法の欠点(つながりの悪いコミュニティができること)を直した、コミュニティ検出の手法です。

実行の間に、**Foundry IQ に対する呼び出し**(`Definition`、`DefinitionUpdate`、`Count`、`RandomSample`、`VectorSampling`、`BatchedLookup`、`Knn`)と、**モデルの呼び出し**(`Model`)が記録されます。

> 出典: [From Louvain to Leiden: guaranteeing well-connected communities(Traag, Waltman, van Eck)](https://arxiv.org/abs/1810.08473) / [From Local to Global: A Graph RAG Approach(Edge et al.)](https://arxiv.org/abs/2404.16130) / [RUN-TOPICGRAPH.md と `topicgraph --help`(v1.0.10 配布物に同梱。チェックポイントの保持と `--resume`)](https://github.com/Azure/foundry-iq-tool-releases/releases/tag/v1.0.10)

> 実環境で確認した障害: 2026-11-19 の Windows イベントログで、`StructureSealer.Seal` のディレクトリ移動がファイル共有違反により終了しました。同じ出力先の `--resume` では、モデルを呼ばずにステージ 6 の完了を確認できました。ロックを保持していたプロセスは未特定です。

In [ ]:
if USE_EXISTING_OUTPUT:
    print("既存出力モード: build は起動しません。次の 5.0 で保存済みデータを読み込みます。")
else:
    summary_path = OUT / "topicGraph" / "build-summary.json"
    build_mode = "--fresh"
    if summary_path.is_file() and json.loads(summary_path.read_text(encoding="utf-8"))["status"] != "completed":
        build_mode = "--resume"

    started = time.time()
    for attempt in range(3):
        build = run_exe("build", "--config", CONFIG_PATH, build_mode, timeout=1800)
        print("\n".join(build.stdout.splitlines()[-25:]), build.stderr)
        if build.returncode == 0:
            break
        sharing_violation = ("System.IO.IOException" in build.stderr and "StructureSealer.Seal" in build.stderr
                             and "being used by another process" in build.stderr)
        if not sharing_violation or attempt == 2:
            build.check_returncode()
        delay = 2 ** (attempt + 1)
        print(f"構造保存時のファイルロックを検出しました。{delay} 秒後に --resume で再試行します。")
        time.sleep(delay)
        build_mode = "--resume"
    elapsed = time.time() - started

    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    calls = summary["calls"]
    if not check("T4-1 build が成功する", summary["status"] == "completed", f"終了コード {build.returncode} / {elapsed:.0f} 秒"):
        raise RuntimeError(f"ビルドが完了していません。{summary_path} と T4 の実行ログを確認してください。")
    check("T4-2 ステージ 0 から 9 がすべて完了または再利用される", [s["number"] for s in summary["stages"]] == list(range(10)) and all(s["status"] in {"completed", "reused"} for s in summary["stages"]),
          [s["number"] for s in summary["stages"] if s["status"] not in {"completed", "reused"}])
    check("T4-3 kNN の呼び出し回数がチャンク数と一致する(全チャンクがアンカー)", calls["Knn"]["calls"] == len(chunks), f"{calls['Knn']['calls']} 回 / チャンク {len(chunks)}")
    check("T4-4 モデルが呼ばれる(ラベル付け)", calls.get("Model", {}).get("calls", 0) >= 1, f"{calls.get('Model', {}).get('calls')} 回")
    check("T4-5 失敗した呼び出しがない", sum(c["failures"] for c in calls.values()) == 0, {k: c["failures"] for k, c in calls.items() if c["failures"]})
    (OUT / "notebook-inputs.json").write_text(json.dumps({"structureId": summary["structureId"], "chunks": chunks}, ensure_ascii=False), encoding="utf-8")
    np.save(OUT / "notebook-vectors.npy", np.asarray(vectors, dtype=np.float32), allow_pickle=False)

### 5.0 保存済み出力を読み込む（再開時の入口）

**1.2 の設定セルの後、このセルから再開できます。** ビルドの完了を確認して、最新リリース、構造、階層、ラベル、チャンク ID と実行記録をディスクから読み込みます。1.2 とこのセルだけで、6.5 のグラフと 6.6 のホイールに進めます。5.1 以降の通常の検証も、この状態を使います。

出力の欠落や未完了ビルドは明示的にエラーにします。新規作成には自動で切り替えません。元チャンクの本文・埋め込みの保存ファイルがない旧ビルドでも、階層・所属 ID・グラフ・ホイールは確認できます。保存ファイルがある場合は、構造 ID、チャンク ID とベクトル形状が今回の出力と一致することも検証します。

> 出典: [Topic Graph のネイティブ実行ファイルと出力](https://github.com/Azure/foundry-iq-tool-releases) / [NumPy load](https://numpy.org/doc/stable/reference/generated/numpy.load.html)

In [ ]:
summary_path = OUT / "topicGraph" / "build-summary.json"
manifest_path = OUT / "topicGraph" / "run-manifest.json"
if not summary_path.is_file() or not manifest_path.is_file():
    raise FileNotFoundError(f"完了済みのビルド出力がありません: {OUT}。出力先を確認してください。未完了ビルドの再開は、1.2 の USE_EXISTING_OUTPUT=False で作成経路を明示的に選びます。")
summary = json.loads(summary_path.read_text(encoding="utf-8"))
if summary["status"] != "completed":
    raise RuntimeError("保存されたビルドは未完了です。T4 の実行ログを確認してください。チェックポイントは削除しないでください。")
calls = summary["calls"]
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
release = next(r for r in manifest["releases"] if r["id"] == manifest["latestRelease"])
structure = next(s for s in manifest["structures"] if s["id"] == release["structureId"])
structure_dir = OUT / "topicGraph" / structure["path"]
hierarchy = json.loads((structure_dir / "hierarchy.json").read_text(encoding="utf-8"))
labels = json.loads((OUT / "topicGraph" / "step-07-labels" / f"{release['labelVariantId']}.json").read_text(encoding="utf-8"))["labels"]
chunk_keys = json.loads((structure_dir / "chunks.json").read_text(encoding="utf-8"))["keys"]
blob = (structure_dir / "leaf-members.bin").read_bytes()
if len(blob) < 12:
    raise ValueError("leaf-members.bin のヘッダーが欠落しています。")
magic, member_version, leaf_count = struct.unpack_from("<4sII", blob, 0)
if magic != b"TGLM" or member_version != 3:
    raise ValueError("leaf-members.bin は TGLM 版 3 の形式である必要があります。")
position, leaf_members = 12, {}
for _ in range(leaf_count):
    if position + 8 > len(blob):
        raise ValueError("leaf-members.bin の所属レコードが途中で切れています。")
    leaf_id, n = struct.unpack_from("<II", blob, position)
    if leaf_id in leaf_members or position + 8 + 4 * n > len(blob):
        raise ValueError("leaf-members.bin の末端 ID が重複しているか、所属データが欠落しています。")
    leaf_members[leaf_id] = list(struct.unpack_from(f"<{n}I", blob, position + 8))
    position += 8 + 4 * n
leaf_counts = {t["id"]: t["memberCount"] for t in hierarchy["topics"] if t["isLeaf"]}
assigned_ordinals = [o for members in leaf_members.values() for o in members]
if (position != len(blob) or set(leaf_members) != set(leaf_counts)
        or any(len(members) != leaf_counts[leaf] for leaf, members in leaf_members.items())
        or sorted(assigned_ordinals) != list(range(len(chunk_keys)))
        or len(set(chunk_keys)) != len(chunk_keys)):
    raise ValueError("チャンクの所属に重複・欠落があるか、トピックの所属数と一致しません。")
context = json.loads((structure_dir / "context.json").read_text(encoding="utf-8"))
chunks, vectors = None, None
inputs_path, vectors_path = OUT / "notebook-inputs.json", OUT / "notebook-vectors.npy"
if inputs_path.exists() != vectors_path.exists():
    raise FileNotFoundError("再開用の本文と埋め込みの保存ファイルが片方だけ存在します。notebook-inputs.json と notebook-vectors.npy を確認してください。")
if inputs_path.is_file():
    saved_inputs = json.loads(inputs_path.read_text(encoding="utf-8"))
    chunks = saved_inputs["chunks"]
    vectors = np.load(vectors_path, allow_pickle=False)
    if (saved_inputs["structureId"] != structure["id"] or len({c["uid"] for c in chunks}) != len(chunks)
            or {c["uid"] for c in chunks} != set(chunk_keys)
            or vectors.shape != (len(chunks), context["dimensions"]) or not np.isfinite(vectors).all()):
        raise ValueError("再開用の本文・埋め込みと、保存されたトピック構造が一致しません。")
else:
    print("元チャンクの本文・埋め込みは未保存です。6.2 はチャンク ID を表示し、6.3 のチャンク PCA は実行しません。再生成は行いません。")
print(f"読み込み完了: リリース {release['id']} / {structure['topics']} トピック / {len(chunk_keys)} 分類済みチャンク")

### 5.1 ステージの時間と、呼び出しの内訳

`build-summary.json` は、ビルドの呼び出しごとに更新される報告書です。ステージごとの所要時間と、呼び出しの種類ごとの回数と時間を、図にします。ソースの本文、ベクトル、認証情報は含まれません。

In [ ]:
stages_df = pd.DataFrame([{"番号": s["number"], "ステージ": s["name"], "状態": s["status"], "実行": s["execution"], "秒": s["elapsedSeconds"], "詳細": s["details"]} for s in summary["stages"]])
calls_df = pd.DataFrame(calls).T[["calls", "attempts", "retries", "failures", "seconds"]]
display(stages_df)
display(calls_df)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].barh(stages_df["番号"].astype(str) + " " + stages_df["ステージ"], stages_df["秒"], color="#6baed6")
axes[0].invert_yaxis()
axes[0].set_title("ステージごとの所要時間(秒)")
axes[1].barh(calls_df.index, calls_df["calls"], color="#fd8d3c")
axes[1].invert_yaxis()
axes[1].set_title("呼び出しの種類ごとの回数")
plt.tight_layout()
plt.show()

## 6. T5: 出力の中身を確かめる

出力は、`outputDirectory` の下に、次の形で残ります。

| 場所 | 内容 |
| --- | --- |
| `topicGraph/run-manifest.json` | 構造とリリースの一覧。最新のリリースの ID と状態 |
| `topicGraph/step-06-structures/<id>/` | 封印された構造。`hierarchy.json`(階層)、`chunks.json`(チャンクのキー)、`leaf-members.bin`(末端ごとの所属)、`statistics.json`、`centroids.f32`(重心) |
| `topicGraph/step-07-labels/<id>.json` | トピックのタイトルと説明(モデルが作ったラベル) |
| `topicGraph/releases/<id>.*` | グラフのスキーマ、文書(`.documents.jsonl`)、オントロジー |
| `ontology/` | 他のツール(`ontology-build`)に渡す、封印された構造とラベルの複製 |

5.0 で読み込んだ最新リリースを検証します。入力チャンク数には、過去の Python 変数ではなく、ビルド時に保存された `context.json` の件数を使います。

T5-3(末端トピックが 2 つ以上)は、`MAX_LEAF_SIZE` とチャンクの数に依存します。小さなコーパスでは、分割されないことがあります。FAIL のときは、`MAX_LEAF_SIZE` か `CHUNK_CHARS` を小さくして、再実行してください。

In [ ]:
check("T5-1 最新のリリースが exported", release["state"] == "exported", release["id"])
check("T5-2 すべてのチャンクがトピックに入る(coverage = 1)", release["coverage"] == 1 and structure["clustered"] == context["documentsAtPreflight"], f"coverage={release['coverage']} / 分類 {structure['clustered']} / 入力 {context['documentsAtPreflight']}")
check("T5-3 末端トピックが 2 つ以上できる", structure["leaves"] >= 2, f"トピック {structure['topics']} / 末端 {structure['leaves']}")
pd.DataFrame({"値": {"リリース": release["id"], "構造": structure["id"][:16], "トピック": structure["topics"], "末端": structure["leaves"], "グラフ文書": release["graphDocumentCount"],
                    "ラベル": release["labeling"]["labels"], "入力トークン": release["labeling"]["inputTokens"], "出力トークン": release["labeling"]["outputTokens"]}})

### 6.1 トピックの階層とラベル

`hierarchy.json` が、トピックの親子(`parent`)、レベル(`level`、0 が最も広い)、所属数(`memberCount`)、末端か(`isLeaf`)を持ちます。ラベルは、別のファイルです。すべてのトピックに、モデルが作ったタイトルが付いたかを確かめます。

In [ ]:
topics_df = pd.DataFrame(hierarchy["topics"])
topics_df["タイトル"] = topics_df["id"].map(lambda i: labels[str(i)]["title"])
topics_df["ラベルの状態"] = topics_df["id"].map(lambda i: labels[str(i)]["status"])
topics_df["説明"] = topics_df["id"].map(lambda i: labels[str(i)]["description"])

check("T5-4 すべてのトピックにタイトルが付く", bool((topics_df["ラベルの状態"] == "ok").all() and (topics_df["タイトル"].str.len() > 0).all()), topics_df["ラベルの状態"].value_counts().to_dict())
topics_df

### 6.2 所属(どのチャンクが、どの末端トピックに入ったか)

`leaf-members.bin` は、末端トピックごとの所属を、チャンクの**序数**で持つバイナリです(版 3 の形式: 識別子 `TGLM`、版、末端の数、そして、末端ごとに「ID、個数、序数の並び」)。`chunks.json` の `keys` が、序数からチャンクのキー(`uid`)への対応です。これは、実行ファイルの**内部形式**を、版 3 を前提に読む手順で、将来の版では変わる可能性があります。

所属の合計が、保存されたチャンク ID の数と一致することを確かめます。本文も保存されている場合はチャンクの冒頭を表示し、旧ビルドなどで本文がない場合は、その旨を明示してチャンク ID を表示します。

In [ ]:
text_by_uid = {c["uid"]: c["snippet"] for c in chunks} if chunks is not None else None
check("T5-5 内部形式(TGLM 版 3)を読め、所属の合計がチャンク数と一致する",
      magic == b"TGLM" and member_version == 3 and sum(len(m) for m in leaf_members.values()) == len(chunk_keys),
      f"末端 {leaf_count} / 所属 {sum(len(m) for m in leaf_members.values())} / チャンク {len(chunk_keys)}")

membership = pd.DataFrame([{"末端": leaf, "タイトル": labels[str(leaf)]["title"], "所属数": len(ordinals),
                            "チャンクの冒頭" if text_by_uid is not None else "チャンク ID（本文未保存）":
                            " | ".join(text_by_uid[chunk_keys[o]][:28].replace("\n", " ") if text_by_uid is not None else chunk_keys[o] for o in ordinals[:3])}
                           for leaf, ordinals in leaf_members.items()])
membership

### 6.3 ベクトル空間での分かれ方を図にする（元の埋め込みが保存されている場合）

元チャンクの埋め込みを、**主成分分析**（PCA）で 2 次元に圧縮し、末端トピックごとに色を付けます。2 次元への圧縮では、元の距離を完全には保てません。

新規ビルドで保存した `notebook-inputs.json` と `notebook-vectors.npy` があれば、カーネル再起動後も同じ入力から描画できます。これらがない旧ビルドでは、このセルは理由を表示して描画を省略します。埋め込みの再生成や Foundry IQ からの再取得は行いません。6.5 のグラフと 6.6 のホイールは、この埋め込みに依存しません。

> 出典: [NumPy の特異値分解](https://numpy.org/doc/stable/reference/generated/numpy.linalg.svd.html)

In [ ]:
if vectors is None:
    print("チャンク PCA は未実施: 元の埋め込みが保存されていません。6.5 と 6.6 の可視化は実行できます。")
else:
    X = np.array(vectors)
    centered = X - X.mean(axis=0)
    _, _, vt = np.linalg.svd(centered, full_matrices=False)
    xy = centered @ vt[:2].T

    leaf_of_uid = {chunk_keys[o]: leaf for leaf, ordinals in leaf_members.items() for o in ordinals}
    leaf_ids = [leaf_of_uid[c["uid"]] for c in chunks]
    fig, ax = plt.subplots(figsize=(9, 6))
    for leaf in sorted(leaf_members):
        mask = np.array([i == leaf for i in leaf_ids])
        ax.scatter(xy[mask, 0], xy[mask, 1], s=60, alpha=0.8, label=f"{leaf}: {labels[str(leaf)]['title'][:18]}")
    ax.set_title("保存されたチャンクのベクトルを PCA で 2 次元に圧縮し、末端トピックで色分け")
    ax.legend(fontsize=8, loc="best")
    plt.show()

### 6.4 リリースの文書(グラフとして書き出されたトピック)

`releases/<id>.documents.jsonl` は、トピックの階層を、**エッジ リスト形式のグラフ文書**(主語、述語、目的語)に書き出したものです。Foundry IQ に公開するときの、そのままの行になります(公開は、プラグインの別のステップです)。述語ごとの件数を見て、文書数がマニフェストの記録と一致することを確かめます。

In [ ]:
documents = [json.loads(line) for line in (OUT / "topicGraph" / release["documentsFile"]).read_text(encoding="utf-8").splitlines() if line.strip()]
check("T5-6 グラフ文書の数がマニフェストの記録と一致する", len(documents) == release["graphDocumentCount"], f"{len(documents)} 件")
pd.DataFrame(documents)["predicate"].value_counts().rename("件数").to_frame()

### 6.5 トピックグラフ全体を表示する

5.0 で読み込んだ `hierarchy` と `labels` を使い、**すべてのトピックと親子関係**を表示します。左端のコレクションは、複数の最上位トピックを束ねる表示用のノードです。矢印は親から子へ向け、右に進むほど細かいトピックになります。タイトルや説明などの属性は、別ノードにせずホバーで確認します。ベクトル自体は描画しません。

- **色**: 最上位トピックごとの枝です。次のホイールでも同じ色を使います。
- **ノードの大きさ**: 所属チャンク数が多いほど大きくします。文書数ではありません。
- **操作**: ホバーで ID・深さ・所属数・説明を確認でき、拡大・移動もできます。配置は階層を表し、埋め込みの距離は表しません。

このセルは `OUT` 配下に **`topic-network.html`** を保存し、ファイルへのリンクを表示します。リンク先を通常のブラウザーで開いてください。Plotly の JavaScript とグラフデータは HTML に内包するため、VS Code の Plotly 専用レンダラー、CDN、インターネット接続は不要です。ノートブック内の埋め込み描画は行いません。

このセルと次のセルは、ローカルで読み込んだデータだけを使います。初回はセクション 1 のインストールセルで `networkx` と `plotly` もインストールしてください。HTML にはトピックのタイトル・説明などが含まれるため、共有する場合は内容を確認してください。

> 出典: [NetworkX の深さ優先探索](https://networkx.org/documentation/stable/reference/algorithms/generated/networkx.algorithms.traversal.depth_first_search.dfs_preorder_nodes.html) / [Plotly のネットワーク図](https://plotly.com/python/network-graphs/) / [Plotly の自己完結型 HTML 出力](https://plotly.com/python/interactive-html-export/)

In [ ]:
from html import escape
from textwrap import wrap
from IPython.display import Markdown, display

import networkx as nx
import plotly.graph_objects as go
from plotly.colors import qualitative

topic_graph = nx.DiGraph()
topic_graph.add_node("collection", title="コレクション", description="分類済みの全チャンク",
                     members=hierarchy["clustered"], parent="", depth=0)
for topic in hierarchy["topics"]:
    topic_id = str(topic["id"])
    parent_id = "collection" if topic["parent"] is None else str(topic["parent"])
    label = labels[topic_id]
    topic_graph.add_node(topic_id, title=label["title"], description=label["description"],
                         members=topic["memberCount"], parent=parent_id)
    topic_graph.add_edge(parent_id, topic_id)

if (not hierarchy["topics"] or len(topic_graph) != len(hierarchy["topics"]) + 1
        or not nx.is_arborescence(topic_graph)):
    raise ValueError("トピック ID の重複、親の欠落、または階層の循環があります。hierarchy を確認してください。")
if hierarchy["clustered"] <= 0:
    raise ValueError("分類済みチャンクがないため、所属割合を可視化できません。")
for topic_id, data in topic_graph.nodes(data=True):
    children = list(topic_graph.successors(topic_id))
    if data["members"] < 0 or (children and sum(topic_graph.nodes[c]["members"] for c in children) != data["members"]):
        raise ValueError(f"トピック {topic_id} の所属数と子トピックの合計が整合しません。")

topic_ids = list(nx.topological_sort(topic_graph))
roots = list(topic_graph.successors("collection"))
color_by_topic = {"collection": "#cbd5e1", **{r: qualitative.Plotly[i % len(qualitative.Plotly)] for i, r in enumerate(roots)}}
for topic_id in topic_ids[1:]:
    parent_id = topic_graph.nodes[topic_id]["parent"]
    topic_graph.nodes[topic_id]["depth"] = topic_graph.nodes[parent_id]["depth"] + 1
    color_by_topic.setdefault(topic_id, color_by_topic[parent_id])

topic_parents = [topic_graph.nodes[t]["parent"] for t in topic_ids]
topic_values = [topic_graph.nodes[t]["members"] for t in topic_ids]
topic_colors = [color_by_topic[t] for t in topic_ids]
topic_titles = [topic_graph.nodes[t]["title"] for t in topic_ids]
topic_hover = [
    f"<b>{escape(data['title'])}</b><br>ID: {t} / 深さ: {data['depth']}<br>所属: {data['members']} チャンク<br>"
    + "<br>".join(escape(line) for line in wrap(data["description"], width=38))
    for t in topic_ids for data in [topic_graph.nodes[t]]
]
leaf_order = [t for t in nx.dfs_preorder_nodes(topic_graph, "collection") if topic_graph.out_degree(t) == 0]
y_by_topic = {t: -float(i) for i, t in enumerate(leaf_order)}
for topic_id in reversed(topic_ids):
    children = list(topic_graph.successors(topic_id))
    if children:
        y_by_topic[topic_id] = float(np.mean([y_by_topic[c] for c in children]))
positions = {t: (topic_graph.nodes[t]["depth"], y_by_topic[t]) for t in topic_ids}
network_fig = go.Figure(go.Scatter(
    x=[positions[t][0] for t in topic_ids], y=[positions[t][1] for t in topic_ids],
    mode="markers+text", textposition="top center",
    text=[("" if t == "collection" else f"ID {t}<br>") + "<br>".join(escape(line) for line in wrap(title, width=14))
          for t, title in zip(topic_ids, topic_titles)],
    hovertext=topic_hover, hovertemplate="%{hovertext}<extra></extra>",
    marker=dict(size=[22 + 38 * np.sqrt(n / hierarchy["clustered"]) for n in topic_values],
                color=topic_colors, line=dict(color="white", width=2)),
))
for parent_id, topic_id in topic_graph.edges:
    network_fig.add_annotation(
        x=positions[topic_id][0], y=positions[topic_id][1],
        ax=positions[parent_id][0], ay=positions[parent_id][1],
        xref="x", yref="y", axref="x", ayref="y", text="", showarrow=True,
        arrowhead=2, arrowwidth=1.3, arrowcolor="#94a3b8", standoff=20, startstandoff=20,
    )
network_fig.update_layout(
    title=f"トピックグラフ全体: {len(topic_ids) - 1} トピック / {topic_graph.number_of_edges()} 親子関係",
    template="plotly_white", height=max(760, 85 * len(leaf_order)), showlegend=False, hovermode="closest",
    font=dict(family="Yu Gothic, Meiryo, sans-serif"), margin=dict(l=80, r=120, t=90, b=50),
    xaxis=dict(visible=False, range=[min(p[0] for p in positions.values()) - 0.5, max(p[0] for p in positions.values()) + 0.5]),
    yaxis=dict(visible=False, range=[min(p[1] for p in positions.values()) - 0.7, max(p[1] for p in positions.values()) + 1.0]),
)
html_header = """<!DOCTYPE html>
<html lang="ja"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Topic Graph</title>
<script>
  (() => {
    const param = new URLSearchParams(window.location.search).get("scoutTheme");
    const theme =
      param || (window.matchMedia("(prefers-color-scheme: dark)").matches ? "dark" : "light");
    document.documentElement.setAttribute("data-theme", theme);
  })();
</script>
<style>
:root {
  color-scheme: light;
  --cp-bg: #f7f4ef;
  --cp-bg-elevated: #fcfbf8;
  --cp-surface: #ffffff;
  --cp-surface-soft: #f5f5f5;
  --cp-border: #dedede;
  --cp-border-strong: #919191;
  --cp-text: #242424;
  --cp-text-muted: #5c5c5c;
  --cp-text-soft: #6f6f6f;
  --cp-accent: #b11f4b;
  --cp-accent-hover: #9a1a41;
  --cp-accent-soft: rgba(177, 31, 75, 0.08);
  --cp-accent-fg: #ffffff;
  --cp-success: #16a34a;
  --cp-danger: #dc2626;
  --cp-warning: #f59e0b;
  --cp-link: #0078d4;
  --cp-shadow: 0 18px 48px rgba(0, 0, 0, 0.12);
  --cp-overlay: rgba(255, 255, 255, 0.8);
  --cp-panel: rgba(255, 255, 255, 0.86);
  --cp-panel-strong: rgba(255, 255, 255, 0.96);
  --cp-sheen: rgba(255, 255, 255, 0.55);
  --cp-highlight: rgba(177, 31, 75, 0.12);
}
html[data-theme="dark"] {
  color-scheme: dark;
  --cp-bg: #3d3b3a;
  --cp-bg-elevated: #343231;
  --cp-surface: #292929;
  --cp-surface-soft: #2e2e2e;
  --cp-border: #474747;
  --cp-border-strong: #5f5f5f;
  --cp-text: #dedede;
  --cp-text-muted: #919191;
  --cp-text-soft: #b0b0b0;
  --cp-accent: #fd8ea1;
  --cp-accent-hover: #fb7b91;
  --cp-accent-soft: rgba(253, 142, 161, 0.14);
  --cp-accent-fg: #1a1a1a;
  --cp-success: #4ade80;
  --cp-danger: #f87171;
  --cp-warning: #fbbf24;
  --cp-link: #4da6ff;
  --cp-shadow: 0 18px 48px rgba(0, 0, 0, 0.32);
  --cp-overlay: rgba(41, 41, 41, 0.88);
  --cp-panel: rgba(41, 41, 41, 0.72);
  --cp-panel-strong: rgba(41, 41, 41, 0.96);
  --cp-sheen: rgba(255, 255, 255, 0.04);
  --cp-highlight: rgba(253, 142, 161, 0.12);
}
body { margin: 0; background: var(--cp-bg); color: var(--cp-text);
       font-family: "Segoe UI", Aptos, Calibri, -apple-system, BlinkMacSystemFont, sans-serif; }
</style></head><body>
"""
html_theme = """
const style = getComputedStyle(document.documentElement);
Plotly.relayout('{plot_id}', {
    paper_bgcolor: style.getPropertyValue('--cp-bg').trim(),
    plot_bgcolor: style.getPropertyValue('--cp-bg').trim(),
    'font.color': style.getPropertyValue('--cp-text').trim(),
    'font.family': 'Segoe UI, Aptos, Calibri, sans-serif'
});
"""
network_html = OUT / "topic-network.html"
network_html.write_text(
    html_header + network_fig.to_html(include_plotlyjs=True, full_html=False,
                                      post_script=html_theme, config={"displaylogo": False}) + "</body></html>",
    encoding="utf-8",
)
display(Markdown(f"[全体グラフをブラウザーで開く]({network_html.as_uri()})"))
print("保存先:", network_html)

### 6.6 階層をトピックホイールで表示する

**6.5 のセルを実行した後**に実行します。中心から、コレクション → トピック階層 → **個々のチャンク**の順に配置します。5.0 で読み込んだ `leaf-members.bin` の所属と `chunks.json` の ID を使い、各チャンクを所属する末端トピックのすぐ外側に並べます。6.2 の実行は不要です。

チャンクは **1 区画 = 1 チャンク**です。区画内の `C1` などは `chunks.json` 内の序数に 1 を加えた表示番号です。ホバーで実際のチャンク ID と所属する末端トピックを確認できます。本文は表示せず、追加の取得も行いません。

チャンクの半径方向の位置は、所属トピックの階層の深さに従います。浅い末端トピックのチャンクは内側に、深い末端トピックのチャンクは外側に表示します。最外周への位置合わせや、そのための透明な区画は追加しません。

**扇形の角度は所属チャンク数に比例**します。親の `memberCount` は子の所属数も含むため、`branchvalues="total"` を指定して二重計上を防ぎます。同じチャンクを各階層で表しているので、全リングの所属数を足してコレクションの件数にはしません。割合の分母は、元文書数ではなく分類済みチャンク数です。

このセルは `OUT` 配下に **`topic-wheel.html`** を保存します。表示されたリンク先を通常のブラウザーで開いてください。全体グラフと同様に JavaScript を内包した単体 HTML なので、Plotly 専用レンダラーや CDN は使いません。

ホバーすると、タイトル・説明・所属数に加えて、**親に対する割合**と**コレクション全体に対する割合**を確認できます。扇形をクリックするとその枝を拡大し、中心をクリックすると上位に戻ります。タイトルが重複しても区別できるように、親子の対応にはタイトルではなく一意のトピック ID を使います。

> 出典: [Plotly のサンバースト図と branchvalues](https://plotly.com/python/sunburst-charts/#branchvalues) / [Sunburst の ids・parents・hovertemplate](https://plotly.com/python/reference/sunburst/)

In [ ]:
wheel_nodes = {
    t: dict(parent=parent, value=value, color=color, hover=hover, kind="topic",
            label=f"コレクション<br>{hierarchy['clustered']} チャンク" if t == "collection" else f"{t}: {escape(title)}")
    for t, parent, value, color, hover, title in zip(topic_ids, topic_parents, topic_values, topic_colors, topic_hover, topic_titles)
}
for leaf, ordinals in leaf_members.items():
    leaf_id = str(leaf)
    for ordinal in ordinals:
        chunk_key = chunk_keys[ordinal]
        hover = (f"<b>チャンク C{ordinal + 1}</b><br>ID: {escape(chunk_key)}"
                 f"<br>所属トピック: {leaf_id}: {escape(topic_graph.nodes[leaf_id]['title'])}"
                 f"<br>所属トピックに対する割合: {1 / len(ordinals):.1%}"
                 f"<br>全体に対する割合: {1 / hierarchy['clustered']:.1%}")
        wheel_nodes[f"chunk:{chunk_key}"] = dict(parent=leaf_id, value=1, color=color_by_topic[leaf_id],
                                                hover=hover, label=f"C{ordinal + 1}", kind="chunk")

wheel_fig = go.Figure(go.Sunburst(
    ids=list(wheel_nodes), parents=[n["parent"] for n in wheel_nodes.values()],
    values=[n["value"] for n in wheel_nodes.values()], labels=[n["label"] for n in wheel_nodes.values()],
    branchvalues="total", sort=False, maxdepth=-1,
    marker=dict(colors=[n["color"] for n in wheel_nodes.values()],
                line=dict(color="white", width=1)),
    insidetextorientation="radial", textinfo="label",
    hovertext=[n["hover"] for n in wheel_nodes.values()],
    hovertemplate=["%{hovertext}<extra></extra>" if n["kind"] == "chunk" else
                   "%{hovertext}<br>親に対する割合: %{percentParent:.1%}<br>全体に対する割合: %{percentRoot:.1%}<extra></extra>"
                   for n in wheel_nodes.values()],
))
wheel_fig.update_layout(
    title=f"トピックホイール: コレクション → トピック階層 → {len(chunk_keys)} チャンク",
    template="plotly_white", height=850, margin=dict(l=20, r=20, t=80, b=20),
    font=dict(family="Yu Gothic, Meiryo, sans-serif"),
)
wheel_html = OUT / "topic-wheel.html"
wheel_html.write_text(
    html_header + wheel_fig.to_html(include_plotlyjs=True, full_html=False,
                                    post_script=html_theme, config={"displaylogo": False}) + "</body></html>",
    encoding="utf-8",
)
display(Markdown(f"トピックホイールの HTML: [ブラウザーで開く]({wheel_html.as_uri()})"))
print("保存先:", wheel_html)

## 7. T6: 一時的な変更が元に戻ること

実行ファイルは、ビルドの間、ソース インデックスに**一時的なスコアリング プロファイル**を追加し(ランダムな標本を取るためと考えられます)、終わると削除します。文書とフィールドは変えません。`profile.json` が、追加した(自分が所有する)プロファイルの状態を記録します。インデックスを読み戻して、次を確かめます。

- プロファイルの状態が、すべて `removed` である。
- インデックスのスコアリング プロファイルの一覧が、ビルド前と同じである。
- 文書数と、ベクトル フィールドの設定(`stored` と `hidden`)が、ビルド前と同じである。

既存出力モードでは、保存された `profile.json` の状態だけを T6-0 として検証します。ビルド前の比較データ `before` がないため、実リソースとの比較である T6-1～3 は未実施と明示します。認証やネットワーク接続は行いません。

失敗や中断で残った場合は、`topicgraph cleanup`(一時的なプロファイルを削除し、ベクトルの取得設定を戻す)で後始末できます。

> 出典: [Plan and apply(foundry-iq-skills)](https://github.com/microsoft/azure-skills/blob/main/.github/plugins/foundry-iq-skills/skills/foundry-iq/workflows/graph/topic-graph-build/references/plan-and-apply.md)

In [ ]:
profile = json.loads((OUT / "topicGraph" / "profile.json").read_text(encoding="utf-8"))
states = [p["state"] for p in profile["profiles"].values()]
if USE_EXISTING_OUTPUT:
    check("T6-0 保存記録で一時プロファイルが削除済みである", bool(states) and all(s == "removed" for s in states), states)
    print("T6-1～3 は未実施: ビルド前の比較データが必要なため、新規作成時だけ実リソースを確認します。")
else:
    after = index_client.get_index(SRC_INDEX)
    after_field = next(f for f in after.fields if f.name == "snippet_vector")
    check("T6-1 一時的なスコアリング プロファイルが削除されている", bool(states) and all(s == "removed" for s in states) and [p.name for p in (after.scoring_profiles or [])] == before["scoring_profiles"], states)
    check("T6-2 文書数が変わらない", search_client.get_document_count() == before["docs"], search_client.get_document_count())
    check("T6-3 ベクトル フィールドの設定が変わらない", (after_field.stored, after_field.hidden) == before["vector_flags"], f"stored={after_field.stored} hidden={after_field.hidden}")

## 8. T7: `build --resume`(完了した build の再利用)

**既存出力モードでは実行しません。** 保存済みファイルの読み込みと、実行ファイルを起動する `build --resume` のテストは別です。以下は新規作成経路だけで確認します。

`--resume` は、確定済みのチェックポイントから再開します。**完了した build に対しては、検証して再利用**し、モデルを再び呼びません。これは、費用と時間を守る性質で、失敗した build を再開するときにも、完了したステージは再計算されません。次を確かめます。

- 終了コードが 0 で、`build-summary.json` の `mode` が `resume`、状態が `completed` である。
- 再開前後の累積モデル呼び出し回数の差分が 0 回である。たとえば、再開前が 12 回、再開後も 12 回なら、追加呼び出しは 0 回です。
- 最新のリリースの ID が、変わらない。

In [ ]:
if USE_EXISTING_OUTPUT:
    print("T7 は未実施: 既存出力を読み込むだけなので、build --resume も起動しません。")
else:
    before_resume = json.loads((OUT / "topicGraph" / "build-summary.json").read_text(encoding="utf-8"))
    model_calls_before = before_resume["calls"].get("Model", {}).get("calls", 0)
    resume = run_exe("build", "--config", CONFIG_PATH, "--resume", timeout=1800)
    print("\n".join(resume.stdout.splitlines()[-8:]), resume.stderr)
    resume.check_returncode()
    summary2 = json.loads((OUT / "topicGraph" / "build-summary.json").read_text(encoding="utf-8"))
    manifest2 = json.loads((OUT / "topicGraph" / "run-manifest.json").read_text(encoding="utf-8"))
    model_calls_after = summary2["calls"].get("Model", {}).get("calls", 0)
    additional_model_calls = model_calls_after - model_calls_before
    check("T7-1 resume が成功し、完了した build を再利用する", resume.returncode == 0 and summary2["status"] == "completed" and summary2["mode"] == "resume",
          f"終了コード {resume.returncode} / mode={summary2['mode']}")
    check("T7-2 モデルを再び呼ばない", additional_model_calls == 0, f"再開前 {model_calls_before} / 再開後 {model_calls_after} / 追加 {additional_model_calls} 回")
    check("T7-3 リリースの ID が変わらない", manifest2["latestRelease"] == manifest["latestRelease"], manifest2["latestRelease"])
    display(pd.DataFrame([{"番号": s["number"], "ステージ": s["name"], "実行": s["execution"], "秒": s["elapsedSeconds"]} for s in summary2["stages"]]))

## 9. Foundry IQ への公開・読み戻し・可視化

**この章も同じノートブックで完結します。** ローカルの完了済みリリースを専用インデックスへ登録し、Foundry IQ から読み戻した実データを描画して GQL で探索します。機能テスト 1・3、モデルの追加設定、Blob Storage は不要です。トピック検索プロファイルやナレッジ ベースへの接続は対象外です。

ローカル出力だけの `Run All` を維持するため、既定の `RUN_FOUNDRY_IQ` は `False` です。公開・読み戻しを行う場合は次の設定セルで `True` にします。初回登録では計画の確認後に `CONFIRM_TOPIC_WRITES = True` として作成セル以降を実行します。

**後日の読み取り:** 新しいカーネルで **9.1 の設定・共通処理 → 9.4 → 9.5** のみ実行できます。`TOPIC_INDEX_NAME`、または保存記録を使うため、ローカルのビルド出力は不要です。先頭の作成用設定も実行しません。

> 出典: [Topic Graph の検索構成と前提](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-topic-graph) / [Documents - Graph](https://learn.microsoft.com/rest/api/searchservice/documents/graph?view=rest-searchservice-2026-11-01-preview)

### 9.1 接続先と実行方法

`RUN_FOUNDRY_IQ = True` で接続します。既存インデックス名を明示した場合は登録処理を飛ばします。名前が空欄なら保存記録、次に `ft-published-topics-` の候補を確認します。候補が複数なら明示的な指定が必要です。候補がない場合だけ初回登録の計画へ進みます。既存リソースは上書き・削除しません。保存記録には接続先・API バージョン・登録名だけを残します。

In [ ]:
import hashlib
import json
import os
import textwrap
import time
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd
from IPython.display import display
from matplotlib.patches import Patch
from azure.identity import DefaultAzureCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import SearchIndex
from dotenv import load_dotenv

import matplotlib
matplotlib.rcParams["font.sans-serif"] = ["Yu Gothic", "Meiryo", "MS Gothic", "Noto Sans CJK JP", "Noto Sans JP", "DejaVu Sans"]

RUN_FOUNDRY_IQ = False
CONFIRM_TOPIC_WRITES = False
TOPIC_INDEX_NAME = ""
TOPIC_OUTPUT = Path(".functional-test") / "topicgraph-out"
PUBLISHED_STATE = Path(".functional-test") / "published-resources.json"
PREFIX = "ft-published"
API_VERSION = "2026-11-01-preview"
MAX_DISPLAY_EDGES = 200

if RUN_FOUNDRY_IQ:
    if not 1 <= MAX_DISPLAY_EDGES <= 1000:
        raise ValueError("MAX_DISPLAY_EDGES は 1 から 1000 の範囲にしてください。")
    load_dotenv(override=True)
    SEARCH_ENDPOINT = os.environ["AZURE_SEARCH_ENDPOINT"].rstrip("/")
    credential = DefaultAzureCredential(exclude_interactive_browser_credential=True)
    index_client = SearchIndexClient(SEARCH_ENDPOINT, credential, api_version=API_VERSION)
else:
    print("9 章は未実施: Foundry IQ への公開・読み取りには RUN_FOUNDRY_IQ = True を指定してください。")

In [ ]:
if RUN_FOUNDRY_IQ:
    if not TOPIC_INDEX_NAME and PUBLISHED_STATE.is_file():
        saved_targets = json.loads(PUBLISHED_STATE.read_text(encoding="utf-8"))
        if saved_targets["endpoint"].rstrip("/") != SEARCH_ENDPOINT or saved_targets["api_version"] != API_VERSION:
            raise ValueError("保存記録と現在の接続先・API バージョンが異なります。登録名または保存先を明示してください。")
        TOPIC_INDEX_NAME = saved_targets.get("topic_index", "")
    if not TOPIC_INDEX_NAME:
        topic_candidates = sorted(name for name in index_client.list_index_names() if name.startswith(f"{PREFIX}-topics-"))
        display(pd.DataFrame({"Topic Graph 候補": topic_candidates}))
        if len(topic_candidates) > 1:
            raise ValueError("TOPIC_INDEX_NAME に既存インデックス名を指定してください。")
        if topic_candidates:
            TOPIC_INDEX_NAME = topic_candidates[0]
    CREATE_TOPIC = not bool(TOPIC_INDEX_NAME)
    print("初回登録の計画へ進みます。" if CREATE_TOPIC else f"登録済みインデックスを読み取ります: {TOPIC_INDEX_NAME}")

### 描画用の共通処理

ノードの色は種類、矢印のラベルは関係です。同じノード間の複数の関係を落とさないよう、`MultiDiGraph` を使います。Topic Graph は親から子へ左から右に配置し、それ以外は読みやすさのための自動配置を使います。距離が意味の近さや信頼度を表すものではありません。

> 出典: [NetworkX MultiDiGraph](https://networkx.org/documentation/stable/reference/classes/multidigraph.html) / [spring_layout](https://networkx.org/documentation/stable/reference/generated/networkx.drawing.layout.spring_layout.html) / [multipartite_layout](https://networkx.org/documentation/stable/reference/generated/networkx.drawing.layout.multipartite_layout.html)

In [ ]:
def draw_graph(graph, title, positions=None):
    if not graph:
        raise ValueError("描画対象の関係がありません。取得件数とオントロジーを確認してください。")
    kinds = sorted({data["kind"] for _, data in graph.nodes(data=True)})
    palette = {kind: plt.cm.tab20(number % 20) for number, kind in enumerate(kinds)}
    if positions is None:
        positions = nx.spring_layout(graph, seed=7, k=1.5)
    styles = [f"arc3,rad={number * 0.12}" for number in range(10)]
    fig, ax = plt.subplots(figsize=(16, 11))
    nx.draw_networkx_nodes(graph, positions, ax=ax, node_size=1800, alpha=0.9,
                           node_color=[palette[graph.nodes[node]["kind"]] for node in graph])
    nx.draw_networkx_edges(graph, positions, ax=ax, arrows=True, arrowsize=15, node_size=1800, connectionstyle=styles, edge_color="#777777")
    nx.draw_networkx_labels(graph, positions, ax=ax, font_size=9,
                            labels={node: textwrap.fill(str(data["label"]), 15) for node, data in graph.nodes(data=True)},
                            bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.8, "pad": 1.5})
    nx.draw_networkx_edge_labels(graph, positions, ax=ax, edge_labels=nx.get_edge_attributes(graph, "predicate"),
                                 font_size=7, rotate=False, node_size=1800, connectionstyle=styles)
    ax.legend(handles=[Patch(color=palette[kind], label=kind) for kind in kinds], loc="upper left")
    ax.set_title(title)
    ax.axis("off")
    plt.tight_layout()
    plt.show()

### GQL の結果を表示する

トピックグラフとエンティティグラフのどちらにも `SearchClient.graph(gql=...)` を使います。対象は `graphProfile.kind = edgeList` のインデックスです。共通の表示処理で結果表、診断、実行統計を確認し、具体的なクエリは 9.5 で実行します。GQL の直接実行では、ナレッジ ベースや Azure OpenAI を呼び出しません。

### 結果の読み方と制約

- `node.id` と `node.name`、`edge.predicate` は、インデックスの `graphProfile.fields` が定める論理的な ID・表示名・関係です。物理フィールド名を GQL に直接書くわけではありません。トピックの `name` は、描画時に `prop:title` から補ったタイトルと異なる場合があります。
- 型・関係・名称で探索範囲を絞ります。このサービスでは、それらを指定しない全エッジ探索は拒否されます。名称は一意とは限らないため、特定の実体を固定したい場合は、取得した `id` で絞り込んでください。
- 各例は `LIMIT 10` で返却行数を制限します。`@odata.count` は返却された行数であり、全グラフの件数ではありません。実行統計や診断も表示するため、推定結果や実行上限に関する警告を確認してください。
- 0 行は、その条件に一致する経路を取得できなかったことを示します。抽出の完了や、元文書に事実が存在しないことの証明にはなりません。

> 出典: [Documents - Graph（要求・結果・診断・統計）](https://learn.microsoft.com/rest/api/searchservice/documents/graph?view=rest-searchservice-2026-11-01-preview) / [Entity Graph の直接 GQL クエリ](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-entity-graph#query-the-graph-directly-with-gql) / [Topic Graph の検索設定](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-topic-graph)

In [ ]:
from azure.search.documents.models import (
    GraphQueryResult,
    GraphQueryProvenanceParameters,
    GraphQueryProvenanceHydrateDocumentsParameters,
)


def display_gql(result: GraphQueryResult) -> None:
    print("返却行数:", len(result.value))
    display(pd.json_normalize(result.as_dict()["value"]))
    if result.diagnostics:
        print("診断（警告・エラー・実行情報）:")
        display(pd.DataFrame([item.as_dict() for item in result.diagnostics]))
    if result.statistics:
        print("実行統計:")
        display(pd.DataFrame([result.statistics.as_dict()]))

In [ ]:
def require_confirmation(confirmed):
    if not confirmed:
        raise RuntimeError("登録計画と費用・閲覧権限を確認し、この章の承認フラグを True にしてください。Azure への書き込みは実行していません。")

In [ ]:
def save_published_target(target_key, target_name):
    targets = {"endpoint": SEARCH_ENDPOINT, "api_version": API_VERSION}
    if PUBLISHED_STATE.is_file():
        targets = json.loads(PUBLISHED_STATE.read_text(encoding="utf-8"))
        if targets["endpoint"].rstrip("/") != SEARCH_ENDPOINT:
            raise ValueError("保存記録と現在の接続先が異なります。PUBLISHED_STATE を別の保存先にしてください。")
        if targets["api_version"] != API_VERSION:
            raise ValueError("保存記録と API バージョンが異なります。互換性を確認するか、PUBLISHED_STATE を別の保存先にしてください。")
    targets[target_key] = target_name
    PUBLISHED_STATE.parent.mkdir(parents=True, exist_ok=True)
    PUBLISHED_STATE.write_text(json.dumps(targets, ensure_ascii=False, indent=2), encoding="utf-8")
    print("読み取り用の登録先を保存しました:", PUBLISHED_STATE.resolve())

## 9.2 リリースの検証

完了したビルドの `exported` リリースだけを対象にします。`schemaFile` と `documentsFile` はリリースに記録されたものを使い、出力フォルダー外を指すパスは拒否します。登録名には成果物のハッシュを含め、別のリリースと混ざらないようにします。

`@search.action` は SDK が設定するため取り除きますが、元の操作が `upload` 以外の場合は止めます。文書数とキーの重複も、書き込み前に確認します。

> 出典: [Topic Graph 配布物のリリース形式](https://github.com/Azure/foundry-iq-tool-releases/releases/tag/v1.0.10) / [Documents - Index](https://learn.microsoft.com/rest/api/searchservice/documents/index?view=rest-searchservice-2026-11-01-preview)

In [ ]:
if RUN_FOUNDRY_IQ and CREATE_TOPIC:
    topic_root = TOPIC_OUTPUT / "topicGraph"
    build_summary = json.loads((topic_root / "build-summary.json").read_text(encoding="utf-8"))
    if build_summary["status"] != "completed":
        raise RuntimeError("機能テスト 2 のビルドが未完了です。T4 で復旧してから実行してください。")
    manifest = json.loads((topic_root / "run-manifest.json").read_text(encoding="utf-8"))
    release = next(item for item in manifest["releases"] if item["id"] == manifest["latestRelease"])
    if release["state"] != "exported":
        raise RuntimeError("最新リリースが exported ではありません。")

    schema_path = topic_root / release["schemaFile"]
    documents_path = topic_root / release["documentsFile"]
    if not all(path.resolve().is_relative_to(topic_root.resolve()) for path in (schema_path, documents_path)):
        raise ValueError("リリースのファイルが Topic Graph の出力フォルダー外を指しています。")
    schema_payload = json.loads(schema_path.read_text(encoding="utf-8"))
    topic_documents = [json.loads(line) for line in documents_path.read_text(encoding="utf-8").splitlines() if line.strip()]
    if not topic_documents or len(topic_documents) != release["graphDocumentCount"]:
        raise ValueError("リリースの文書数と documentsFile の内容が一致しません。")
    if any(document.get("@search.action") != "upload" for document in topic_documents):
        raise ValueError("upload 以外の操作を含むリリースは登録しません。")
    topic_documents = [{key: value for key, value in document.items() if key != "@search.action"} for document in topic_documents]
    topic_hash = hashlib.sha256(schema_path.read_bytes() + documents_path.read_bytes()).hexdigest()
    TOPIC_INDEX_NAME = f"{PREFIX}-topics-{topic_hash[:12]}"
    topic_index = SearchIndex({**schema_payload, "name": TOPIC_INDEX_NAME, "description": f"graph-test-2 Topic Graph sha256:{topic_hash}"})
    if topic_index.graph_profile is None or topic_index.graph_profile.kind != "edgeList":
        raise ValueError("リリースに edgeList のグラフ プロファイルがありません。")
    key_fields = [field.name for field in topic_index.fields if field.key]
    if len(key_fields) != 1 or len({document[key_fields[0]] for document in topic_documents}) != len(topic_documents):
        raise ValueError("スキーマのキーが一意でないか、文書キーが重複しています。")
    display(pd.DataFrame([{"リリース": release["id"], "文書数": len(topic_documents), "登録先": TOPIC_INDEX_NAME, "SHA-256": topic_hash}]))

## 9.3 Foundry IQ への登録

### 容量と登録計画を確認する

この章で作成するトピック用インデックス 1 個の空き枠だけを確認します。他のノートブックのリソースやモデル設定は不要です。容量確認から作成までの間に、他の処理が容量を使う可能性はあります。

> 出典: [Foundry IQ Service Statistics](https://learn.microsoft.com/rest/api/searchservice/get-service-statistics/get-service-statistics?view=rest-searchservice-2026-11-01-preview) / [サービスの上限](https://learn.microsoft.com/azure/search/search-limits-quotas-capacity)

In [ ]:
if RUN_FOUNDRY_IQ and CREATE_TOPIC:
    topic_counter = index_client.get_service_statistics().counters.index_counter
    if topic_counter is None or topic_counter.quota is None:
        raise RuntimeError("インデックスの容量上限を確認できません。書き込み前に管理者へ確認してください。")
    display(pd.DataFrame([{"対象": "インデックス", "使用中": topic_counter.usage, "上限": topic_counter.quota, "今回必要": 1}]))
    if topic_counter.quota - topic_counter.usage < 1:
        raise RuntimeError("トピックグラフの登録に必要な容量が足りません。既存リソースは自動削除しません。")
    display(pd.DataFrame([
        {"操作": "トピックグラフのインデックス作成と文書登録", "対象": TOPIC_INDEX_NAME, "内容": f"{len(topic_documents)} 文書"},
        {"操作": "使用する API", "対象": SEARCH_ENDPOINT, "内容": API_VERSION},
    ]))

### トピックグラフの書き込みを有効にする

**次の作成・アップロードセルから Azure への書き込みです。** プレビュー API を使用します。接続先、登録名、文書数、登録先の閲覧権限、Foundry IQ の利用料金を確認し、承認する場合だけ `CONFIRM_TOPIC_WRITES = True` に変更してください。この操作はエンティティグラフの書き込みを有効にしません。

同名のインデックスがあれば作成 API のエラーで止まります。更新・削除は行いません。読み取りだけの場合は、このセルと作成・アップロードセルを実行せず、9.4 へ進みます。

### インデックスの作成と文書登録

リリースの `graphProfile`、ベクトル フィールド、検索設定を維持したインデックスを作ります。空のインデックスを作っただけでは完了ではありません。続くセルで文書ごとの登録結果を確認し、反映を待って件数を照合します。

> 出典: [Indexes - Create](https://learn.microsoft.com/rest/api/searchservice/indexes/create?view=rest-searchservice-2026-11-01-preview) / [Documents - Index](https://learn.microsoft.com/rest/api/searchservice/documents/index?view=rest-searchservice-2026-11-01-preview)

In [ ]:
if RUN_FOUNDRY_IQ and CREATE_TOPIC:
    require_confirmation(CONFIRM_TOPIC_WRITES)
    registered_topic_index = index_client.create_index(topic_index)
    print("作成しました:", registered_topic_index.name)

In [ ]:
if RUN_FOUNDRY_IQ and CREATE_TOPIC:
    require_confirmation(CONFIRM_TOPIC_WRITES)
    topic_client = SearchClient(SEARCH_ENDPOINT, TOPIC_INDEX_NAME, credential, api_version=API_VERSION)
    uploaded = 0
    for start in range(0, len(topic_documents), 100):
        batch_results = topic_client.upload_documents(topic_documents[start:start + 100])
        failures = [{"キー": result.key, "状態": result.status_code, "エラー": result.error_message} for result in batch_results if not result.succeeded]
        if failures:
            display(pd.DataFrame(failures))
            raise RuntimeError("文書の登録に失敗しました。上のエラーを解決してから同じ文書を再送してください。")
        uploaded += len(batch_results)
    print("登録を受け付けた文書数:", uploaded)

In [ ]:
if RUN_FOUNDRY_IQ and CREATE_TOPIC:
    topic_client = SearchClient(SEARCH_ENDPOINT, TOPIC_INDEX_NAME, credential, api_version=API_VERSION)
    for attempt in range(30):
        topic_count = topic_client.get_document_count()
        if topic_count == len(topic_documents):
            break
        time.sleep(2)
    else:
        raise TimeoutError(f"文書数の反映が完了しません。期待 {len(topic_documents)} / 実際 {topic_count}")
    print("PASS: Foundry IQ の文書数とリリースが一致:", topic_count)

### トピックグラフの登録先を保存する

件数の一致を確認した後、接続先、API バージョン、トピック用インデックス名を `PUBLISHED_STATE` に保存します。エンティティグラフが未登録でも保存できます。認証情報や文書本文は保存しません。後日は 9.1 → 9.4 から読み戻せます。

In [ ]:
if RUN_FOUNDRY_IQ and CREATE_TOPIC:
    save_published_target("topic_index", TOPIC_INDEX_NAME)

## 9.4 登録済みデータの読み取りと可視化

以下の描画に使うのは `SearchClient.search()` の結果です。描画対象は最大 `MAX_DISPLAY_EDGES` 文書で、全件数も表示します。トピックの属性も 1 文書として数えます。大きなリリースで表示を制限すると、一部のタイトルや親子関係が表示対象外になることがあります。

登録名が未指定で候補が複数ある場合は、`TOPIC_INDEX_NAME` に表示された名前を指定し、この読み取りセルから再実行してください。9.3 のアップロードセルや件数照合セルは不要です。

> 出典: [Indexes - List](https://learn.microsoft.com/rest/api/searchservice/indexes/list?view=rest-searchservice-2026-11-01-preview) / [Documents - Foundry IQ](https://learn.microsoft.com/rest/api/searchservice/documents/search-post?view=rest-searchservice-2026-11-01-preview)

In [ ]:
if RUN_FOUNDRY_IQ:
    if not TOPIC_INDEX_NAME:
        topic_candidates = sorted(name for name in index_client.list_index_names() if name.startswith(f"{PREFIX}-topics-"))
        display(pd.DataFrame({"Topic Graph 候補": topic_candidates}))
        if len(topic_candidates) != 1:
            raise ValueError("Topic Graph を一意に選べません。TOPIC_INDEX_NAME に既存インデックス名を指定してください。")
        TOPIC_INDEX_NAME = topic_candidates[0]
    live_topic_index = index_client.get_index(TOPIC_INDEX_NAME)
    TF = live_topic_index.graph_profile.fields
    topic_client = SearchClient(SEARCH_ENDPOINT, TOPIC_INDEX_NAME, credential, api_version=API_VERSION)
    topic_select = [field.name for field in live_topic_index.fields if field.retrievable is not False]
    topic_found = topic_client.search("*", select=topic_select, include_total_count=True, top=MAX_DISPLAY_EDGES)
    topic_total = topic_found.get_count()
    topic_frame = pd.DataFrame(list(topic_found))
    if topic_frame.empty:
        raise RuntimeError("登録先の Topic Graph から文書を取得できませんでした。")
    print(f"Foundry IQ 内 {topic_total} 文書 / 取得 {len(topic_frame)} 文書")
    display(topic_frame[[TF.subject_id_field, TF.predicate_field, TF.object_id_field, TF.object_string_field]].head(12))

In [ ]:
if RUN_FOUNDRY_IQ:
    topic_links = topic_frame[topic_frame[TF.predicate_field].isin(["HasRootTopic", "HasSubTopic"])]
    topic_titles = topic_frame[topic_frame[TF.predicate_field] == "prop:title"].set_index(TF.subject_id_field)[TF.object_string_field].to_dict()
    topic_names = topic_frame.set_index(TF.subject_id_field)[TF.subject_id_display_name_field].to_dict()
    topic_graph = nx.from_pandas_edgelist(topic_links, TF.subject_id_field, TF.object_id_field,
                                        edge_attr=TF.predicate_field, create_using=nx.MultiDiGraph())
    nx.set_edge_attributes(topic_graph, {(a, b, key): data[TF.predicate_field] for a, b, key, data in topic_graph.edges(keys=True, data=True)}, "predicate")
    for node in topic_graph:
        topic_graph.nodes[node].update(label=topic_titles.get(node, topic_names.get(node, node)), kind="Topic" if node in topic_titles else "TopicGraph")
    levels = {node: level for level, nodes in enumerate(nx.topological_generations(topic_graph)) for node in nodes}
    nx.set_node_attributes(topic_graph, levels, "level")
    positions = nx.multipartite_layout(topic_graph, subset_key="level", align="vertical")
    draw_graph(topic_graph, f"Foundry IQ 登録済み Topic Graph: {TOPIC_INDEX_NAME}", positions=positions)

## 9.5 GQL による階層探索・集計

9.1 の GQL 表示処理と、9.4 の読み取りセルを実行してから進みます。描画セルと他のノートブックは不要です。ルート、親子の経路、親ごとの接続数を順に確認します。これはトピックグラフの多様性を考慮したベクトル検索設定とは別の操作です。

> 出典: [Documents - Graph](https://learn.microsoft.com/rest/api/searchservice/documents/graph?view=rest-searchservice-2026-11-01-preview) / [Topic Graph の検索構成](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-topic-graph)

### 9.5.1. ルート直下のトピック

9.4 の読み取りセルが作った `topic_client` を使います。`MATCH` でルートからトピックへの矢印を指定し、`WHERE` で `HasRootTopic` に絞ります。`RETURN` は結果表の列、`ORDER BY` は並び順です。この例の `graph_id` はルートノードの ID です。

> 出典: [Documents - Graph](https://learn.microsoft.com/rest/api/searchservice/documents/graph?view=rest-searchservice-2026-11-01-preview)。`HasRootTopic` と `HasSubTopic` は、この Notebook が取り込んだ Topic Graph リリースの関係名です。

In [ ]:
if RUN_FOUNDRY_IQ:
    topic_roots_gql = """
    MATCH (root)-[edge]->(topic)
    WHERE edge.predicate = 'HasRootTopic'
    RETURN root.id AS graph_id, topic.id AS topic_id, topic.name AS topic_name
    ORDER BY topic_id
    LIMIT 10
    """
    topic_roots_result = topic_client.graph(
        gql=topic_roots_gql, include_statistics=True, include_diagnostics=True,
    )
    display_gql(topic_roots_result)

### 9.5.2. 階層を 2 ホップたどる

2 本の矢印が同じ `parent` ノードを共有する経路を取得します。ルートとの関係は `HasRootTopic`、親子間の関係は `HasSubTopic` です。独立した 2 回の検索結果を Python 側で結合するのではなく、GQL の 1 クエリで経路を指定します。

> 出典: [複数ホップの GQL パターン](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-entity-graph#query-the-graph-directly-with-gql) / [Documents - Graph](https://learn.microsoft.com/rest/api/searchservice/documents/graph?view=rest-searchservice-2026-11-01-preview)

In [ ]:
if RUN_FOUNDRY_IQ:
    topic_path_gql = """
    MATCH (root)-[first]->(parent)-[second]->(child)
    WHERE first.predicate = 'HasRootTopic' AND second.predicate = 'HasSubTopic'
    RETURN root.id AS graph_id, parent.name AS parent_topic, child.name AS child_topic
    ORDER BY parent_topic, child_topic
    LIMIT 10
    """
    topic_path_result = topic_client.graph(
        gql=topic_path_gql, include_statistics=True, include_diagnostics=True,
    )
    display_gql(topic_path_result)

### 9.5.3. 親トピックごとの接続数

`HasSubTopic` の経路を親の ID・表示名ごとに集計します。同じ表示名を持つ別ノードを混ぜないため、`GROUP BY` に ID も含めます。`COUNT(*)` は一致した経路の行数であり、インデックス内の文書総数ではありません。元のグラフに重複した関係がある場合、重複排除済みの子ノード数と同じとは限りません。

> 出典: [Documents - Graph（GQL の実行と返却行数）](https://learn.microsoft.com/rest/api/searchservice/documents/graph?view=rest-searchservice-2026-11-01-preview)。以下の集計構文は、この Notebook のプレビュー API と実データで検証します。

In [ ]:
if RUN_FOUNDRY_IQ:
    topic_count_gql = """
    MATCH (parent)-[edge]->(child)
    WHERE edge.predicate = 'HasSubTopic'
    RETURN parent.id AS parent_id, parent.name AS parent_topic, count(*) AS child_connections
    GROUP BY parent.id, parent.name
    ORDER BY child_connections DESC, parent_id
    LIMIT 10
    """
    topic_count_result = topic_client.graph(
        gql=topic_count_gql, include_statistics=True, include_diagnostics=True,
    )
    display_gql(topic_count_result)

### 読み取り結果と後片付け

以下は 9.4 の読み取り結果です。トピック用インデックスへの登録と、ナレッジ ベースから検索できるように構成することは別です。この Notebook は検索プロファイルの接続を行いません。

不要になった場合は、表示されたトピック用インデックスの所有者・依存関係を確認してから削除してください。この章の公開先は末尾の後片付けでは削除しません。機能テスト 2 の元の成果物や共有リソースは削除しません。

> 出典: [Topic Graph の検索構成](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-topic-graph)

In [ ]:
if RUN_FOUNDRY_IQ:
    display(pd.DataFrame([
        {"成果物": "トピックグラフ", "登録先": TOPIC_INDEX_NAME, "確認": f"Foundry IQ 内 {topic_total} 文書"},
    ]))

## 10. 結果の一覧

テスト ID(`T4-1` など)ごとに最新の結果だけを採用し、PASS と FAIL の件数と詳細を表示します。セルを再実行した場合やテスト名の説明が変わった場合も、同じ ID の過去の結果は集計しません。再実行後も失敗しているテストは FAIL として残ります。

In [ ]:
summary_df = pd.DataFrame(results)
test_ids = summary_df["テスト"].str.split(n=1).str[0]
summary_df = summary_df.loc[~test_ids.duplicated(keep="last")].reset_index(drop=True)
print(summary_df["結果"].value_counts().to_dict())
summary_df.style.apply(lambda row: ["background-color: #ffd6d6" if row["結果"] == "FAIL" else "" for _ in row], axis=1)

## 11. 後片付け(任意)

**既存出力モードでは、`RUN_CLEANUP=True` でも削除しません。** 以下は新規作成経路で後片付けを明示的に有効にした場合の動作です。

テスト専用のインデックス `ft-topic-src` と、作業フォルダー(出力、ダウンロードした実行ファイル)を削除します。**機能テスト 3 の入力として出力を使うときは、先に実行しないでください**。削除の前に、`topicgraph cleanup` で、一時的なプロファイルが残っていないことも確かめます。

In [ ]:
if USE_EXISTING_OUTPUT:
    print("既存出力モード: インデックスと作業フォルダーの削除は行いません。")
elif RUN_CLEANUP:
    print(run_exe("cleanup", "--config", CONFIG_PATH, timeout=600).stdout[-500:])
    index_client.delete_index(SRC_INDEX)
    shutil.rmtree(WORK, ignore_errors=True)
    print("削除しました:", SRC_INDEX, WORK)